In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from riotwatcher import RiotWatcher, LolWatcher, ApiError

In [ ]:
load_dotenv(dotenv_path="../.env", override=True)
api_key = os.getenv("RIOT_API_KEY")

riot_watcher = RiotWatcher(api_key)
lol_watcher = LolWatcher(api_key)

ACCOUNT_REGION = "asia"
PLATFORM_REGION = "kr"
MATCH_REGION = "asia"

riot_id_name = "Hide on bush"
riot_id_tag = "KR1"

print("Watchers initialized")

In [ ]:
account_data = riot_watcher.account.by_riot_id(ACCOUNT_REGION, riot_id_name, riot_id_tag)
puuid = account_data["puuid"]

summoner_data = lol_watcher.summoner.by_puuid(PLATFORM_REGION, puuid)

print("PUUID fetched successfully")
print("Summoner Level:", summoner_data["summonerLevel"])

In [ ]:
try:
    match_ids = lol_watcher.match.matchlist_by_puuid(MATCH_REGION, puuid, count=5)
    print("Recent match IDs:")
    for i, match_id in enumerate(match_ids, start=1):
        print(f"{i}. {match_id}")
except ApiError as err:
    print(f"API error: {err.response.status_code}")

In [ ]:
target_match_id = match_ids[0]
print("Target match:", target_match_id)

try:
    match_detail = lol_watcher.match.by_id(MATCH_REGION, target_match_id)
    print("Match detail fetched successfully")
    print("Game Mode:", match_detail["info"]["gameMode"])
    print("Game Duration:", match_detail["info"]["gameDuration"])
except ApiError as err:
    print(f"API error: {err.response.status_code}")

In [ ]:
player_row = None

for participant in match_detail["info"]["participants"]:
    if participant["puuid"] == puuid:
        player_row = participant
        break

if player_row:
    print("Champion:", player_row["championName"])
    print("Win:", player_row["win"])
    print("Kills:", player_row["kills"])
    print("Deaths:", player_row["deaths"])
    print("Assists:", player_row["assists"])
    print("Gold Earned:", player_row["goldEarned"])
else:
    print("Player not found in match participants")

In [ ]:
summary_df = pd.DataFrame([
    {
        "match_id": target_match_id,
        "game_mode": match_detail["info"]["gameMode"],
        "game_duration": match_detail["info"]["gameDuration"],
        "champion": player_row["championName"],
        "win": player_row["win"],
        "kills": player_row["kills"],
        "deaths": player_row["deaths"],
        "assists": player_row["assists"],
        "gold_earned": player_row["goldEarned"],
        "total_damage_to_champions": player_row["totalDamageDealtToChampions"],
    }
])

summary_df

In [ ]:
import os
os.makedirs("../data/interim", exist_ok=True)
print("interim folder ready")

In [ ]:
summary_df.to_csv("../data/interim/match_summary_sample.csv", index=False)
print("Saved to ../data/interim/match_summary_sample.csv")